# PIPELINE ROBUSTO DE MACHINE LEARNING CON AUDITORÍA E MEJORA DE IA
**Asignatura:** Proyecto Integrador de Machine Learning  
**Entregable:** Cuaderno para Google Classroom / Canvas (.ipynb)  
**Fecha:** Noviembre 2026  

---

## 1. Importación de Librerías y Configuración del Entorno

In [1]:
import numpy as np
import pandas as pd
import logging
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Configuración de Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
print('Librerías cargadas exitosamente.')

## 2. Generación y Carga del Dataset Sintético de Evaluación

In [2]:
# Simulación de Dataset de Clientes (Sintético con nulos y variables mixtas)
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'Edad': np.random.choice([np.nan, 25, 30, 35, 45, 50, 60], size=n_samples),
    'Ingreso_Anual': np.random.choice([np.nan, 30000, 45000, 60000, 85000, 120000], size=n_samples),
    'Puntuacion_Credito': np.random.randint(300, 850, size=n_samples),
    'Tipo_Cuenta': np.random.choice(['Ahorro', 'Corriente', 'Inversión'], size=n_samples),
    'Estado_Civil': np.random.choice(['Soltero', 'Casado', 'Divorciado'], size=n_samples),
    'Fuga_Cliente': np.random.choice([0, 1], size=n_samples, p=[0.7, 0.3])
})

print('Dimensiones del Dataset:', data.shape)
data.head()

## 3. Partición Estricta de Datos (Prevención de Data Leakage)
Atendiendo la **Nota Importante**: La división se realiza antes de cualquier proceso de imputación o escalamiento.

In [3]:
X = data.drop(columns=['Fuga_Cliente'])
y = data['Fuga_Cliente']

# Partición Estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape}, Test set: {X_test.shape}')

## 4. Construcción del Pipeline Robusto con ColumnTransformer

In [4]:
# Definición de Columnas
num_cols = ['Edad', 'Ingreso_Anual', 'Puntuacion_Credito']
cat_cols = ['Tipo_Cuenta', 'Estado_Civil']

# Preprocesamiento Numérico
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocesamiento Categórico
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Pipeline Completo
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

print('Pipeline de ML configurado correctamente.')

## 5. Entrenamiento, Búsqueda de Hiperparámetros y Evaluación

In [5]:
# Mapeo de Parámetros para GridSearch
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    full_pipeline, param_grid, cv=cv_strategy, scoring='f1', n_jobs=-1
)

# Entrenamiento atómico
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print('Mejores Parámetros:', grid_search.best_params_)

# Evaluación en conjunto de prueba independiente
y_pred = best_model.predict(X_test)

print('\n=== REPORTE DE CLASIFICACIÓN FINAL ===')
print(classification_report(y_test, y_pred))
print('Matriz de Confusión:\n', confusion_matrix(y_test, y_pred))

## 6. Persistencia del Modelo y Prueba de Inferencia

In [6]:
# Guardar modelo entrenado
model_filename = 'pipeline_fuga_clientes.joblib'
joblib.dump(best_model, model_filename)
print(f'Modelo guardado exitosamente como {model_filename}')

# Carga y prueba de inferencia con una muestra nueva
loaded_model = joblib.load(model_filename)
nueva_muestra = pd.DataFrame([{
    'Edad': 38,
    'Ingreso_Anual': 75000,
    'Puntuacion_Credito': 680,
    'Tipo_Cuenta': 'Corriente',
    'Estado_Civil': 'Casado'
}])

prediccion = loaded_model.predict(nueva_muestra)
probabilidad = loaded_model.predict_proba(nueva_muestra)[:, 1]

print(f'Predicción para nuevo cliente: {"Fuga" if prediccion[0] == 1 else "Permanencia"}')
print(f'Probabilidad de Fuga: {probabilidad[0]:.2%}')